In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.dates import year_fraction
from lib.import_data import load_market, MarketEngine
from lib.vanilla_option import OptionType, VanillaOptionBlackSholes, d,strike_for_delta, vanna_volga_cost_coefficients
from lib.binary_option import CoNBlackScholes, AoNBlackScholes
from lib.digital_option import OneTouchBlackScholes, NoTouchBlackScholes, PaymentType, BoundType, theta, vartheta, d as d_digi

class PricingEngine:
    def __init__(self, market_file: str, tenor: str, imply_pln: bool):
        self.market = load_market(market_file, tenor)
        self.m_engine = MarketEngine(self.market)

        self.r_pln, self.r_eur, self.df_d, self.df_f = (self.m_engine.rates_dfs(imply_pln))

        self.sigma_atm = self.market.atm
        self.sigma_25C = self.market.atm + self.market.bf25 + 0.5 * self.market.rr25
        self.sigma_25P = self.market.atm + self.market.bf25 - 0.5 * self.market.rr25
        
    def strike(self,delta: float,option_type: OptionType):
        sigma = self.sigma_atm
        return strike_for_delta(
            delta=delta,
            option_type=option_type,
            df_d=self.df_d,
            df_f=self.df_f,
            S_t=self.market.spot,
            sigma=sigma,
            t=self.market.start,
            T=self.market.expiry,
            base=365,
            forward=self.market.delta_forward,
            premium=self.market.delta_premium,
        )
         
    def vanilla_price(self,K: float,option_type: OptionType,pricing_method: str,p: float = 1.0,):
        option = VanillaOptionBlackSholes(T=self.market.expiry,K=K,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def con_price(self,K: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001,):
        option = CoNBlackScholes(T=self.market.expiry, K=K, option_type=option_type )
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )
        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def aon_price(self,K: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001):
        option = AoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def one_touch_price(self,B :float ,option_type: OptionType, bound_type: BoundType, payment_type: PaymentType, pricing_method: str = "bs",p: float = 1.0):
        option = OneTouchBlackScholes(T=self.market.expiry,B=B,bound_type=bound_type, payment_type=payment_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
            )
    def no_touch_price(self,B :float ,option_type: OptionType, bound_type: BoundType, pricing_method: str = "bs",p: float = 1.0):
        option = NoTouchBlackScholes(T=self.market.expiry,B=B,bound_type=bound_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
            )

    
    def option_price_range(self, product_type:str, option_type: OptionType, K: float, pricing_method: str, 
                           bound_type: BoundType,payment_type: PaymentType, n: int = 100, b: float=0.05):
        S0 = self.market.spot
        spots = np.linspace(S0 *(1-b), S0 *(1+b), n)
        prices = []

        match product_type:
            case "Vanilla":
                option = VanillaOptionBlackSholes(T=self.market.expiry,K=K,option_type=option_type)
            case "CoN":
                option = CoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
            case "AoN":
                option = AoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
            case "OneTouch":
                option = OneTouchBlackScholes(T=self.market.expiry,B=K,bound_type=bound_type,payment_type=payment_type)     
            case "NoTouch":
                option = NoTouchBlackScholes(T=self.market.expiry,B=K,bound_type=bound_type) 
            case _:
                raise ValueError(f"Nieznany typ opcji: {product_type}")
        for S in spots:
            
            if pricing_method=="bs":
                price = option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=S,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
                )
            elif pricing_method == "vv":
                if product_type=="Vanilla" or product_type=="OneTouch" or product_type=="NoTouch" :
                    price = option.vanna_volga_price(
                    df_d=self.df_d,
                    df_f=self.df_f,
                    p=1.0,
                    S_t=S,
                    sigma_atm=self.sigma_atm,
                    sigma_25C=self.sigma_25C,
                    sigma_25P=self.sigma_25P,
                    t=self.market.start,
                    delta_forward=self.market.delta_forward,
                    delta_premium=self.market.delta_premium,
                    base=365,)
                elif product_type=="CoN" or product_type=="AoN":
                    price = option.vanna_volga_price(
                    df_d=self.df_d,
                    df_f=self.df_f,
                    p=1.0,
                    S_t=S,
                    sigma_atm=self.sigma_atm,
                    sigma_25C=self.sigma_25C,
                    sigma_25P=self.sigma_25P,
                    t=self.market.start,
                    delta_forward=self.market.delta_forward,
                    delta_premium=self.market.delta_premium,
                    base=365,
                    dK=0.0001
                    ) 
            else:
                raise ValueError("Metoda to nie BS ani VV")

            prices.append(price)
        return spots, np.array(prices)
    
    
    def plot_vs_spot(self,product_type:str,K: float, option_type: OptionType=OptionType.CALL, bound_type:BoundType=BoundType.LOWER_BOUND,payment_type:PaymentType= PaymentType.AT_MATURITY ):
        
        spots, bs = self.option_price_range(product_type, option_type, K,bound_type=bound_type,payment_type=payment_type,pricing_method= "bs",)
        x, vv = self.option_price_range(product_type, option_type, K, bound_type=bound_type,payment_type=payment_type, pricing_method="vv")
        
        if product_type=="OneTouch" or product_type=="NoTouch":
            plt.figure()

            plt.plot(spots, bs, label="Black-Scholes")
            plt.plot(spots, vv, label="Vanna-Volga")

            plt.axvline(self.market.spot, linestyle="--", label="Bariera")

            plt.xlabel("FX Spot")
            plt.ylabel("Cena Opcji")

            if product_type=="OneTouch":
                plt.title(f"Cena opcji vs Spot dla {product_type} {bound_type.name} {payment_type.name}")
            else:
                plt.title(f"Cena opcji vs Spot dla {product_type} {bound_type.name}")
            plt.legend()
            plt.show()
        else:
            plt.figure()

            plt.plot(spots, bs, label="Black-Scholes")
            plt.plot(spots, vv, label="Vanna-Volga")

            plt.axvline(self.market.spot, linestyle="--", label="Spot")

            plt.xlabel("FX Spot")
            plt.ylabel("Cena Opcji")
        
            plt.title(f"Cena opcji vs Spot dla {product_type} {option_type.name}")
            plt.legend()
            plt.show()

<class 'ModuleNotFoundError'>: No module named 'pandas'

In [2]:
engine = PricingEngine("market_data.xlsx", tenor="3M",imply_pln=True)
vanilla = engine.vanilla_price(
    K=engine.strike(0.25, OptionType.CALL),
    option_type=OptionType.CALL,
    pricing_method="vv"
)
print(engine.strike(0.25, OptionType.CALL))
con = engine.con_price(
    K=engine.strike(0.25, OptionType.CALL),
    option_type=OptionType.CALL,
    pricing_method="vv"
)

aon = engine.aon_price(
    K=engine.strike(0.25, OptionType.CALL),
    option_type=OptionType.CALL,
    pricing_method="vv"
)

no = engine.no_touch_price(
    B=4,
    option_type=OptionType.CALL,
    bound_type=BoundType.LOWER_BOUND,
    pricing_method="vv"
)

one = engine.one_touch_price(
    B=4,
    option_type=OptionType.CALL,
    bound_type=BoundType.LOWER_BOUND,
    payment_type=PaymentType.AT_MATURITY,
    pricing_method="vv"
)

print("Vanilla:", vanilla)
print("Cash-or-Nothing:", con)
print("Asset-or-Nothing:", aon)
print(f"No touch down at B={4}:, payment at maturity",no)
print(f"One touch down at B={4}: ",one)

engine.plot_vs_spot(K=engine.strike(0.25, OptionType.PUT),product_type="AoN",option_type=OptionType.CALL)
engine.plot_vs_spot(K=4.3,product_type="OneTouch",bound_type=BoundType.LOWER_BOUND,payment_type=PaymentType.AT_MATURITY)
engine.plot_vs_spot(K=4.3,product_type="OneTouch",bound_type=BoundType.UPPER_BOUND,payment_type=PaymentType.AT_MATURITY)

<class 'NameError'>: name 'PricingEngine' is not defined